# 0️⃣1️⃣ - Managing your Remote Users
![Persona](https://img.shields.io/badge/%F0%9F%91%A4%20Persona-%F0%9F%9B%A0%EF%B8%8F%20Platform%20%2F%20IT%20Administrator-lightgrey) ![Difficulty](https://img.shields.io/badge/%F0%9F%9A%A6%20Difficulty-Beginner-green) ![RADKit version](https://img.shields.io/badge/RADKit-1.9.6-blue?logo=cisco&logoColor=white) ![Python version](https://img.shields.io/badge/Python-3.12%2B-yellow?logo=python&logoColor=white)

> In this playbook we will explore how to manage your service's remote users: retrieving, creating, updating and deleting users.

---

### 📋 What you'll learn

| # | Topic |
|---|-------|
| 1 | How to connect to a RADKit service using the `ControlAPI` context manager |
| 2 | How to list all remote users and retrieve a specific user by username |
| 3 | How to create a new remote user with a configurable activation window |
| 4 | How to update the attributes of an existing remote user |
| 5 | How to delete a remote user |

### 🛠️ Before You Begin

If you haven't set up your environment yet, follow the instructions in [SETUP.md](../SETUP.md) before running any cells.

---

### 🤖 The `ControlAPI` Object

The `ControlAPI` object is your main entry point for administering a RADKit service programmatically. It acts as an alternative to the WebUI, giving you full control over service components — devices, remote users, labels, admins, and the service itself — over the network.

Used as a context manager, it authenticates as a service superadmin and returns a live handle through which all subsequent API calls are made. The example below shows how to create a connection and confirm the type of object returned:

In [19]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    print(type(service))

<class 'radkit_service.control_api.ControlAPI'>


---

## 👤 Method 1: Retrieve Remote Users

**Best for:** Auditing the current user roster, pre-flight checks before automation runs, or pulling the exact attribute state of a specific user.

**How it works:**
1. Connect to the service with `ControlAPI.create()`.
2. Call `list_remote_users()` to get all configured users, or `get_remote_user(username=...)` for a targeted single-user lookup.
3. Each result is a `StoredRemoteUser` Pydantic model instance containing all user attributes.

**What you need:**
- The RADKit server address and superadmin password

### 1.1 List all users and inspect a specific one

In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    
    # A list of TBD class instances is returned
    remote_users = service.list_remote_users().result
    print(f"\n👥 There are {len(remote_users)} remote users configured in this service:\n")
    
    for remote_user in remote_users:
        u = remote_user.username
        masked = u[:3] + "*" * (u.index("@") - 3) + u[u.index("@"):]
        print(f"👤 Username: {masked}")

    # Getting the information of a user. Ever since we know that a single match will be returned, we get the first element of the result list.
    my_user = service.get_remote_user(username="radkit_chatops.gen@cisco.com").result[0]
    print(f"\n-------------------\n➡️ About user radkit_chatops.gen@cisco.com:")
    print(f"🪪 Full username: {my_user.fullname}")
    print(f"📝 Description: {my_user.description}")
    print(f"🕓 {"This user doesn't expire" if my_user.timeSliceMinutes is None else f'Time slice: {my_user.timeSliceMinutes} minutes'}")
    print(f"☁️  Cloud connection: {'✅ enabled' if my_user.connectionModeCloudActive else '❌ disabled'}")
    print(f"↔️  Direct connection: {'✅ enabled' if my_user.connectionModeDirectActive else '❌ disabled'}")
    print(f"🔑 Direct + cloud identity: {'✅ enabled' if my_user.connectionModeDirectSsoActive else '❌ disabled'}")
    


👥 There are 11 remote users configured in this service:

👤 Username: lst*****@cisco.com
👤 Username: sao*********@cisco.com
👤 Username: nsi*****@cisco.com
👤 Username: alf*****@cisco.com
👤 Username: rad***************@cisco.com
👤 Username: mgo*****@cisco.com
👤 Username: mbo*****@cisco.com
👤 Username: pgr*****@cisco.com
👤 Username: tch*****@cisco.com
👤 Username: eri*****@cisco.com
👤 Username: tes*****@acme.com

-------------------
➡️ About user radkit_chatops.gen@cisco.com:
🪪 Full username: RADKit ChatOps
📝 Description: RADKit ChatOps
🕓 This user doesn't expire
☁️  Cloud connection: ✅ enabled
↔️  Direct connection: ✅ enabled
🔑 Direct + cloud identity: ✅ enabled


---

## ➕ Method 2: Create a Remote User

**Best for:** Automating user provisioning: bootstrapping a new engineer's access, spinning up a temporary account for a demo, or scripting onboarding as part of a CI/CD pipeline.

**How it works:**
1. Call `create_remote_user()` with the desired username, display name, description, and connection modes.
2. Set the activation window using one of three helpers: `ActivateForever()`, `ActivateMinutes(n)`, or `Deactivate()`.
3. Confirm creation by fetching the new user with `get_remote_user()`.

**What you need:**
- The RADKit server address and superadmin password
- A valid email-format username not already present in the service

### 2.1 Create a user with a time-limited activation

In [2]:
from getpass import getpass
from radkit_client import CustomSecretStr
from radkit_service.control_api import ControlAPI, ActivateMinutes

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    
    service.create_remote_user(
        username = "test.gen@acme.com",
        full_name = "Test ACME",
        description = "Test ACME User",
        activate = ActivateMinutes(5), # When not set, it defaults to ActivateForever()
        connection_mode_cloud_active = False,
        connection_mode_direct_active = True,
        connection_mode_direct_sso_active = False
    )
    
    # Now, let's fetch the new user we just created to verify its information
    my_user = service.get_remote_user(username="test.gen@acme.com").result[0]
    print(f"➡️ About user test.gen@acme.com:")
    print(f"🪪 Full username: {my_user.fullname}")
    print(f"📝 Description: {my_user.description}")
    print(f"🕓 {"This user doesn't expire" if my_user.timeSliceMinutes is None else f'Time slice: {my_user.timeSliceMinutes} minutes'}")
    print(f"☁️  Cloud connection: {'✅ enabled' if my_user.connectionModeCloudActive else '❌ disabled'}")
    print(f"↔️  Direct connection: {'✅ enabled' if my_user.connectionModeDirectActive else '❌ disabled'}")
    print(f"🔑 Direct + cloud identity: {'✅ enabled' if my_user.connectionModeDirectSsoActive else '❌ disabled'}")
    

➡️ About user test.gen@acme.com:
🪪 Full username: Test ACME
📝 Description: Test ACME User
🕓 Time slice: 5 minutes
☁️  Cloud connection: ❌ disabled
↔️  Direct connection: ✅ enabled
🔑 Direct + cloud identity: ❌ disabled


---

## ✏️ Method 3: Update a Remote User

**Best for:** Changing user attributes after provisioning: extending an expiring access window, toggling connection modes, or refreshing the display name and description.

**How it works:**
1. Call `update_remote_user()` with the target username and any fields you want to change.
2. Fields you omit retain their current values.
3. Confirm the changes by fetching the user again with `get_remote_user()`.

> **Note:** The username itself cannot be changed. To rename a user, delete and recreate them.

In [ ]:
from getpass import getpass
from radkit_service.control_api import ControlAPI, ActivateForever

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    
    service.update_remote_user(
        "test.gen@acme.com",
        full_name = "Test ACME Updated",
        description = "Test ACME User - Updated",
        activate = ActivateForever(),
        connection_mode_cloud_active = True,
        connection_mode_direct_active = True,
        connection_mode_direct_sso_active = True
    )
    
    # Now, let's fetch the new user we just created to verify its information
    my_user = service.get_remote_user(username="test.gen@acme.com").result[0]
    print(f"➡️ About user test.gen@acme.com:")
    print(f"🪪 Full username: {my_user.fullname}")
    print(f"📝 Description: {my_user.description}")
    print(f"🕓 {"This user doesn't expire" if my_user.timeSliceMinutes is None else f'Time slice: {my_user.timeSliceMinutes} minutes'}")
    print(f"☁️  Cloud connection: {'✅ enabled' if my_user.connectionModeCloudActive else '❌ disabled'}")
    print(f"↔️  Direct connection: {'✅ enabled' if my_user.connectionModeDirectActive else '❌ disabled'}")
    print(f"🔑 Direct + cloud identity: {'✅ enabled' if my_user.connectionModeDirectSsoActive else '❌ disabled'}")
    

➡️ About user test.gen@acme.com:
🪪 Full username: Test ACME Updated
📝 Description: Test ACME User - Updated
🕓 This user doesn't expire
☁️  Cloud connection: ✅ enabled
↔️  Direct connection: ✅ enabled
🔑 Direct + cloud identity: ✅ enabled


---

## 🗑️ Method 4: Delete a Remote User

**Best for:** Removing access for a decommissioned user, cleaning up temporary accounts after a demo, or rotating identities as part of an automation workflow.

**How it works:**
1. Call `delete_remote_user(username=...)` with the exact username to remove.
2. Check `.root.success` on the returned `APIResult` to confirm the operation succeeded.

> **Bulk option:** Pass a list of usernames to `delete_remote_users()` to remove multiple users in one call.

### 4.1 Delete a user and confirm the result

In [12]:
from getpass import getpass
from radkit_service.control_api import ControlAPI

radkit_server = input("👤 Enter your RADKit server IP address: ")
radkit_password = getpass("🔑 Please input the password of your RADKit server: ")

with ControlAPI.create(
    base_url=f"https://{radkit_server}:8081/api/v1",
    admin_name="superadmin",
    admin_password=radkit_password,
    http_client_kwargs=dict(verify=False),
) as service:
    
    result = service.delete_remote_user(
        username = "test.gen@acme.com"
    )
    
    print(f"👤 User test.gen@acme.com deleted: {result.root.success}")

👤 User test.gen@acme.com deleted: True


---

## 🔀 Which Method Should I Use?

| | 👤 Retrieve | ➕ Create | ✏️ Update | 🗑️ Delete |
|---|---|---|---|---|
| **Function** | `list_remote_users()` / `get_remote_user()` | `create_remote_user()` | `update_remote_user()` | `delete_remote_user()` |
| **Bulk variant** | `list_remote_users()` returns all | `create_remote_users()` | `update_remote_users()` | `delete_remote_users()` |
| **Return type** | `StoredRemoteUser` | — | — | `APIResult` |
| **Username changeable** | — | Set at creation | ❌ No | — |
| **Best for** | Auditing, pre-flight checks | Provisioning, demos | Expiry, mode toggles | Decommission, cleanup |